In [1]:
import pandas as pd
import numpy as np
import re
import string

In [2]:
df = pd.read_parquet('data/train-00000-of-00001.parquet')

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 4254 entries, 0 to 4253
Data columns (total 9 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   id                   4254 non-null   str   
 1   local_id             4254 non-null   str   
 2   specialty            4254 non-null   str   
 3   author               4254 non-null   str   
 4   reviewer             4254 non-null   str   
 5   pool                 4254 non-null   str   
 6   suggested_scenario   4254 non-null   object
 7   documents            4254 non-null   object
 8   structured_abstract  4254 non-null   object
dtypes: object(3), str(6)
memory usage: 539.0+ KB


## 1 - Eclatement de la colonne documents contenant un dictionnaire en une colonne par cle du dictionnaire

In [4]:
df = pd.concat([
        df.drop(columns='documents'), 
        pd.json_normalize(df['documents'])], 
    axis=1)

## 2 - Separation des colonnes (type, header, text, word_count) par ordre

In [5]:
cols = ['type', 'header', 'text']

lengths = df[cols].apply(lambda col: col.apply(len))

In [6]:
(lengths.nunique(axis=1) == 1).all()

np.True_

In [7]:
df.loc[3400, 'type']

array(['CRH'], dtype=object)

In [8]:
len(df.loc[3400, 'type'])

1

In [9]:
cols_to_explode = ['type', 'header', 'text', 'word_count']

# adding indexes to columns with more than 1 type
df['_suffix'] = df[(df['type']).apply(len) > 1]['type'].apply(
    lambda x: np.arange(1, len(x) + 1)
)

In [10]:
df['_suffix'].sample(10)

3993          NaN
934           NaN
3677          NaN
2654          NaN
1416    [1, 2, 3]
2445          NaN
282           NaN
2546          NaN
1591    [1, 2, 3]
3369          NaN
Name: _suffix, dtype: object

In [11]:
# exploding the columns simultaneously
df = df.explode(cols_to_explode + ['_suffix'], ignore_index=True)

# mask that selects lines with o suffixes
mask = df['_suffix'].notna()

suffix = '_' + df['_suffix'].astype(str)

df.loc[mask, 'id'] = df.loc [mask, 'id'].astype(str) + suffix
df.loc[mask, 'local_id'] = df.loc[mask, 'local_id'].astype(str) + suffix

df = df.drop(columns=['_suffix'])

In [12]:
lengths = df[cols].apply(lambda col: col.apply(len))
(lengths.nunique(axis=1) == 1).all()

np.False_

In [13]:
df['word_count'] = df['word_count'].astype(int)

In [14]:
print(f'Nombres de lignes contenants des textes de plus de 512 mots de base : {df[df['word_count'] > 512].shape[0]}')

Nombres de lignes contenants des textes de plus de 512 mots de base : 3101


In [15]:
df.dtypes

id                        str
local_id                  str
specialty                 str
author                    str
reviewer                  str
pool                      str
suggested_scenario     object
structured_abstract    object
type                      str
header                    str
text                      str
word_count              int64
dtype: object

## 3 - Exploration du corpus (Spécialités et longueurs)
Aperçu de la répartition des documents par service médical et analyse statistique de la longueur des textes bruts.

In [16]:
df['specialty'].value_counts()

specialty
CHIRURGIE VISCERALE           849
CHIR ORTHO ET TRAUMATO        680
GYNECOLOGIE                   594
PNEUMOLOGIE                   531
CANCERO ADULTE                507
UROLOGIE                      497
ANATOMOPATHOLOGIE             438
OBSTETRIQUE                   322
MALADIES INFECTIEUSES         285
NEUROLOGIE                    251
MEDECINE INTERNE              228
HEPATO-GASTRO-ENTERO          170
MEDECINE GERIATRIQUE          155
HEMATOLOGIE CLINIQUE          153
CHIR.CARDIO-VASC.             117
NEPHROLOGIE                   104
CARDIOLOGIE                    96
MEDECINE INTER-SPECIALITES     82
MEDECINE PEDIATRIQUE           82
REANIMATION                    44
Name: count, dtype: int64

In [17]:
df['word_count'].describe()

count    6185.000000
mean      639.069846
std       504.442240
min        24.000000
25%       297.000000
50%       514.000000
75%       823.000000
max      5370.000000
Name: word_count, dtype: float64

### Faisabilité
les modeles BERT son tlimites a 512 tokens donc dans notre cas ce n'est pas possible avant d'effectuer quelque traitement afin de reduire nos textes (suppression de la ponctuation, stopwords ...)

## 4 - Nettoyage du texte et validation technique
Suppression de la ponctuation et des espaces multiples pour assainir les données. On recalcule ensuite la taille des documents nettoyés pour vérifier combien dépassent la limite stricte de 512 tokens imposée par les modèles type CamemBERT.

In [18]:
# remplacer la ponctuation par des espaces
df['text'] = df['text'].astype(str).str.replace(rf'[{string.punctuation}]', ' ', regex=True)

In [19]:
def clean_spaces(text: str) -> str:
        text = re.sub(r'\s+', ' ', text) # enlève tous les espaces en trop (\n, \t, \r, \v...)
        return text.strip()
df['text'] = df['text'].apply(clean_spaces)

In [20]:
df['word_count'] = df['text'].str.split().str.len()

In [21]:
df['word_count'].describe()

count    6185.000000
mean      450.224091
std       333.237084
min        20.000000
25%       223.000000
50%       367.000000
75%       581.000000
max      3481.000000
Name: word_count, dtype: float64

In [47]:
print(f'Nombres de lignes contenants des tesxtes de plus de 512 mots après suppression des éspaces et de la ponctuation : {df[df['word_count'] > 512].shape[0]}')

Nombres de lignes contenants des tesxtes de plus de 512 mots après suppression des éspaces et de la ponctuation : 1942


## 5 - Filtrage des documents pertinents (Mentions du tabac)
Utilisation d'expressions régulières (regex) pour identifier tous les comptes rendus qui abordent le sujet du tabac.

In [ ]:
# expressions régulières selon les mots-clés de la doc des M1
re_tabac = (
    r'\btabac\b|\btabagisme|\btabagique|intoxication tabagique|'
    r'(alcoolo|éthylo|ethylo)[- ]?tabagi|post[- ]?tabagi|'
    r'\bfumeu(r|se)s?\b|\bnon[- ]?fumeu(r|se)|\b(ancien(ne)?|ex)[- ]?fumeu(r|se)|'
    r'jamais\s+fum|\bfum(e|ait|aient|er|é)\b|\bcigarettes?\b|\bcig\b|\bcigares?\b|'
    r'cigarettes? électronique|\bvapot|e-cigarette|\bla pipe\b|nicotin|'
    r'\d\s*PA\b|\bpaquets?\b|paquets?[- ]?ann[ée]es?|\bsevr|sevrage tabagique|tabacolog'
)

re_tabac = (
    r"\btabac\b|\btabagisme|\btabagique|intoxication tabagique|"
    r"(?:alcoolo|éthylo|ethylo)[- ]?tabagi|post[- ]?tabagi|"
    r"\bfumeu(?:r|se)s?\b|\bnon[- ]?fumeu(?:r|se)|\b(?:ancien(?:ne)?|ex)[- ]?fumeu(?:r|se)|"
    r"jamais\s+fum|\bfum(?:e|ait|aient|er|é)\b|\bcigarettes?\b|\bcig\b|\bcigares?\b|"
    r"cigarettes? électronique|\bvapot|e-cigarette|\bla pipe\b|nicotin|"
    r"\d\s*PA\b|\bpaquets?\b|paquets?[- ]?ann[ée]es?|\bsevr|sevrage tabagique|tabacolog"
)

# regex_complements = r'\b(patchs?|gommes?)\b' (105 lignes en plus)

df['mention_tabac'] = df['text'].str.contains(re_tabac, case=False, regex=True, na=False)

nombre_docs_tabac = df['mention_tabac'].sum()
print(f"Documents avec mention du tabac : {nombre_docs_tabac}")

C:\Users\Ce PC\AppData\Local\Temp\ipykernel_13896\1286963659.py:13: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df['mention_tabac'] = df['text'].str.contains(re_tabac, case=False, regex=True, na=False)


Documents avec mention du tabac : 2780


## 6 - Modèle de référence : Règle à mots-clés (Version 1)
Création d'une première approche de classification basée sur des conditions strictes, de la plus certaine (termes de négation forts) à la plus ambiguë. L'objectif est d'évaluer cette règle sur un échantillon pour en identifier les failles.

In [49]:
def regle_de_base(texte):
    texte = str(texte).lower()
    # non-fumeur, jamais fumé
    if re.search(r'\b(non[- ]?fumeu(?:r|se)|jamais\s+fum)', texte):
        return "Non-fumeur"
    # ancien fumeur, ex-fumeur, sevré, sevrage tabagique, post-tabagique
    elif re.search(r'\b(?:ancien(?:ne)?|ex)[- ]?fumeu(?:r|se)|\bsevr|sevrage tabagique|post[- ]?tabagi', texte):
        return "Ancien fumeur"
    # intoxication tabagique, alcoolo-tabagique, PA (paquets-années), la pipe, vapot    
    # tabagisme, tabagique, fumeur, cigarette, cigare
    elif re.search(r'intoxication tabagique|(?:alcoolo|éthylo|ethylo)[- ]?tabagi|\d\s*PA\b|\bpaquets?\b|\bla pipe\b|\bvapot|e-cigarette|\btabagisme|\btabagique|\bfumeu(?:r|se)s?\b|\bcigarettes?\b|\bcigares?\b', texte):
        return "Fumeur actuel"
    else:
        return "Non mentionné"

In [50]:
echantillon = df[df['mention_tabac'] == True].sample(5, random_state=42)

for index, row in echantillon.iterrows():
    statut_predit = regle_de_base(row['text'])
    print(f"id :{index}")
    print(f"text : {row['text']}...")
    print(f"prédiction de la règle : {statut_predit}\n")

id :4018
text : Compte rendu d hospitalisation Service de Médecine gériatrique Monsieur Christophe Giner Correas âgé de 77 ans a été hospitalisé en service de Médecine Gériatrique du 03 11 2025 au 17 11 2025 Motif d hospitalisation Admis depuis les urgences pour majoration de dyspnée oedèmes des membres inférieurs et altération de l état général évoluant depuis plusieurs jours Antécédents médicaux chirurgicaux familiaux allergies Antécédents médicaux • Hypertension artérielle ancienne • Fibrillation auriculaire paroxystique suivie en cardiologie • Insuffisance cardiaque chronique à FEVG modérée • Maladie rénale chronique stade 3a • Syndrome parkinsonien idiopathique diagnostiqué il y a 4 ans • Trouble cognitif majeur d allure Alzheimer • Diabète de type 2 traité par insuline • Hypotension orthostatique intermittente • Carence en vitamine B9 découverte en 2023 Antécédents chirurgicaux • Cholécystectomie laparoscopique en 1998 • Hernie inguinale droite opérée en 2015 Allergies connues • 

## Bilan du test : 
La règle marche bien pour les mots simples, mais elle se trompe dès qu'il y a une négation (comme "pas de tabagisme" ou "non tabagique"). Elle voit le mot lié au tabac et classe directement le patient en fumeur, sans comprendre le sens de la phrase. Cela prouve qu'une simple recherche de mots-clés ne suffit pas pour des comptes rendus médicaux.

## 7 - Création du jeu d'entraînement et pré-annotation
Exclusion des 60 documents déjà utilisés par les M1, puis tirage aléatoire reproductible (seed fixe) de 120 nouveaux documents équilibrés (60 avec mention du tabac, 60 sans). La règle de base est ensuite appliquée sur ce jeu pour pré-annoter automatiquement le statut de chaque patient.

In [51]:
# exclusion des documents des M1
ids_m1 = [] 
df_filtre = df[~df['id'].isin(ids_m1)]

# 60 documents avec mention du tabac et 60 autres sans mention
echantillon_positif = df_filtre[df_filtre['mention_tabac'] == True].sample(60, random_state=42)
echantillon_negatif = df_filtre[df_filtre['mention_tabac'] == False].sample(60, random_state=42)

df_annot_1 = pd.concat([echantillon_positif, echantillon_negatif])

# mélange du jeu 
df_annot_1 = df_annot_1.sample(frac=1, random_state=42).reset_index(drop=True)

# pré-annotation automatique avec la règle de base
df_annot_1['pre_annotation'] = df_annot_1['text'].apply(regle_de_base)

print(df_annot_1['pre_annotation'].value_counts())

pre_annotation
Non mentionné    71
Ancien fumeur    26
Fumeur actuel    20
Non-fumeur        3
Name: count, dtype: int64
